In [6]:
import os 
base_path = '/mnt/c/Users/harsh/OneDrive/Documents/course/thesis/testing_hardwicke/framework-reproducibility/framework_sample_run' # add the path to your folder with all data and framework script here
os.chdir(base_path)
print(os.getcwd())


/mnt/c/Users/harsh/OneDrive/Documents/course/thesis/testing_hardwicke/framework-reproducibility/framework_sample_run


In [7]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR))

import data_snippet       # summarizer helper
import config             # loads dotenv keys


from config import OPENAI_API_KEY, ANTHROPIC_API_KEY

In [8]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("API key not found.")
else:
    print("API key found.")

API key found.


In [9]:
api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    print("API key not found.")
else:
    print("API key found.")

API key found.


In [ ]:
from anthropic import Anthropic
from openai import OpenAI
import pypdf

client = OpenAI()
checker = Anthropic()

In [5]:
import pypdf
# Function to extract text from specific pages of a PDF
def extract_pdf_text(pdf_path, start_page, end_page):
    with open(pdf_path, "rb") as f:
        reader = pypdf.PdfReader(f)
        extracted_text = ""
        # Note: page indices start at 0
        for i in range(start_page-1, end_page):
            extracted_text += reader.pages[i].extract_text() + "\n"
    return extracted_text

In [ ]:
# Provide path for the main article
article_path = 'article.pdf'

#Select the start and end page from the article you require
start_page = 2
end_page = 5

# Page extractor
pdf_text = extract_pdf_text(article_path, start_page, end_page)

print(pdf_text)

##### Loading the require datasets
    -Have to edit dataset to have 1 heading column  (can be tried to fix)

In [ ]:

#----------------------------------------------------------------------------------------------------------------#
# Loading and processing the dataset - depends on how data is structured and what format it is in.
# This section is highly dependent on the dataset format and structure.
# + 
# Specific data cleaning and processing steps for the dataset
#----------------------------------------------------------------------------------------------------------------#
import pandas as pd
import glob
import pyreadstat                      # Uncomment to read .sav / spss files
import numpy as np
import docx2txt
import json

dataset, meta = pyreadstat.read_sav('data/BogusVisualFeedbackData.sav') # Add path where the original dataset is saved
print(dataset.head())

######### If codebook for the dataset is available, load it here

codebook = meta.column_names_to_labels  # Convert the labels to a string format
codebook = json.dumps(codebook, indent=2)
#codebook = codebook.replace("\n\n", "\n")
print(codebook) 


# Save the dataset as loaded to ensure final code works well.
dataset.to_csv('dataset.csv', index=False)

In [12]:
#----------------------------------------------------------------------------------------------------------------#
# Creating a snippet of the dataset for LLM processing
# +
# Add any additional context regarding dataset and compile into a message for LLM processing
#----------------------------------------------------------------------------------------------------------------#
from data_snippet import make_llm_snippets

dataset_preview, dataset_summary, cat_col = make_llm_snippets(
    file_path = 'dataset.csv',
    id_cols = None,
    extra_strata = [],
    n_per_stratum = 5
)

add_dataset_context = """
The dataset provided is a sample from the main dataset used in the experiment.
It contains 5 random observations out of 48 total observations. 
Below provided is some additional context about the dataset apart from the codebook:
- There are 3 variables that do not have any values in the dataset. The variables are: 
    - 'NormalisedDataUploadedToDataverse'
    - 'RawData'
    - 'NormalisedDataNotUploaded'
- These variables are explained in the codebook, and they define the set of three variables that are placed after each of the above variables in the dataset. 
"""

In [13]:
print(dataset_preview)
print(dataset_summary)
print(cat_col)

### Dataset sample – Random sample of 5 rows (no grouping)

|   Participant |   DirectionofRotation |   NormalisedDataUploadedToDataverse |   Condition1_Gain0.8 |   Condition2_Gain1 |   Condition3_Gain1.2 |   RawData |   Point8 |   One |   Onepoint2 |   NormalisedDataNotUploaded |   Point.8 |   One1 |   One1.2 |
|---------------|-----------------------|-------------------------------------|----------------------|--------------------|----------------------|-----------|----------|-------|-------------|-----------------------------|-----------|--------|----------|
|            17 |                     1 |                                 nan |                 0.89 |                  1 |                 0.79 |       nan |    35.31 | 39.89 |       31.43 |                         nan |      0.99 |   1.12 |     0.88 |
|            13 |                     1 |                                 nan |                 0.84 |                  1 |                 0.97 |       nan |    51.74 | 61.58 | 

In [12]:
import base64

# select total number of images / figre:
tot_img = 1

# Provide path for plots / figure / images
#   Format for providing path
#   path_1 = <path_of_fig_1>
#   path_2 = <path_of_fig_2>
#   ....

path_1 = 'figures/figure_2.png'


########## Automated code starts here ##########

path_list_fig = []

for i in range(tot_img):
    path_list_fig.append(globals()[f'path_{i+1}'])

# Function to encode image to base64
def encode_image(path):
    with open(path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


encode_list_fig = {}

for i, path in enumerate(path_list_fig):
    encode_list_fig[f'figure_{i+1}'] = encode_image(path)

Function to create model response for each figure

In [13]:
def create_model_response(encode_figure):
    response_figure = client.chat.completions.create(
        model = 'gpt-4o',
        messages = [
            {
                'role': 'user',
                'content': [
                    {"type": "text", 'text': "Provide a detailed description of the graph / figure provided in the image which can be easily understood by another Large language model, allowing it to reimagine the whole figure. The other LLM can not see the image, so the description should be very detailed."},
                    {
                        "type": "image_url",
                        "image_url": {
                            'url': f"data:image/png;base64,{encode_figure}"
                        }
                    }
                ]
            }
        ]
    )

    return response_figure.choices[0].message.content

# Create a dictionary of model responses for each figure:
model_responses = {}
for figure_name, encode_data in encode_list_fig.items():
    model_responses[figure_name] = create_model_response(encode_data)

In [ ]:
print(model_responses['figure_1'])

In [13]:
import base64
# select total number of tables:
tot_tab = 1

# Provide path for plots / figure / images
#   Format for providing path
#   path_1 = <path_of_fig_1>
#   path_2 = <path_of_fig_2>
#   ....

path_1_tab = 'figure/table_2.png'
#path_2_tab = 'images/fig_s1.png'
#path_3_tab = 'images/supMat_1.png'

########## Automated code starts here ##########

path_list_tables = []

for i in range(tot_tab):
    path_list_tables.append(globals()[f'path_{i+1}_tab'])

# Function to encode image to base64
def encode_image(path):
    with open(path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


encode_list_tables = {}

for i, path in enumerate(path_list_tables):
    encode_list_tables[f'table_{i+1}'] = encode_image(path)

In [ ]:
## Table Extraction - not needed here
def create_table_response(table_image):
    response_table = client.chat.completions.create(
        model = 'gpt-4o',
        messages = [
            {
                'role': 'user',
                'content': [
                    {"type": "text", 'text': "Extract the table from the image and convert to a csv format, readable by a Large Language Model clearly.  Also add the note for the table from the image to the output."},
                    {
                        "type": "image_url",
                        "image_url": {
                            'url': f"data:image/png;base64,{table_image}"
                        }
                    }
                ]
            }
        ]
    )
    return response_table.choices[0].message.content

# Create a dictionary of model responses for each table:
model_responses_table = {}
for table_name, encode_data in encode_list_tables.items():
    model_responses_table[table_name] = create_table_response(encode_data)


In [ ]:
print(model_responses_table['table_1'])

In [ ]:
# Prompt for the first stage:

dev_job = ('You are a reproducibility editor for a scientific journal, well trained in statistical analysis, your goal is to replicate the code for the analysis ' # you are a reproducibility editor for a scientific jounral, your sole goal is to reprordice the analysis of the article using the provided excerpt to you
            'as per the excerpt from provided article. The main motivation behind the research is not important to you, only the analysis '
            'process is important. '
            'Focus on the target results asked for when creating the analysis process. Do not worry about the full article.')

# ---------- PIPELINE INSTRUCTIONS ---------- #

pipeline_inst = ('Pipeline should contain a step by step process of exactly how the complete analysis process is suggested in the article. This pipleine should consider the following: \n ' #refer article_ref instead of article, additionally put a cut-off for any extra analysis suggested by article or model basis the reference
                 '1. import_libraries – list required packages  '
                 '2. read_data – load the datasets '
                 '3. clean_and_recode – handle missing data, type casting, relabeling variables per article '
                 '4. isolate_target_subset – keep rows/columns relevant to the focal analysis (conditions/tasks) '
                 '5. descriptive_summary – compute the summary measures as asked for in Reference excerpt (means, frequencies, etc.) '
                 '6. verify_assumptions – run only the assumption checks explicitly mentioned in the article '
                 '7. run_primary_analysis – execute the statistical test(s) the authors describe with their stated parameters '
                 '8. calculate_reported_effect_size – derive the effect-size metric the authors report (e.g., Cohen’s d, η², odds ratio) '
                 '9. compile_results – gather the exact figures the paper presents for tables/text.'
                 'Make sure to only have 1 set of numbering for the steps, and do not repeat any step. '
                 'Do not use any placeholders like "e.g." or "etc." in the steps, every step should be a concrete R function. ')

# Specific instructions for what part of analysis is to be reproduced.
hw_outcome_specs = (f"""For this article you should focus on the findings reported in section 'Primary outcome: pain-free range of
motion'

Specifically, you should attempt to reproduce all descriptive and inferential analyses reported in the text below and associated tables/figures:\n""")
article_ref = (f"""
> The repeated measures ANOVA revealed a large overall
effect of visual-proprioceptive feedback (condition) on
pain-free range of motion F(2, 94) = 18.9, p < .001, ηp2 = 0.29. All pairwise comparisons were significant (ps < .01). As shown in Figure 3, when vision understated true rotation,
pain-free range of motion was increased, and this
was a medium-sized effect, p = .006, d = 0.67; when
vision overstated true rotation, pain-free range of motion was decreased, and this was a large effect, p = .001, d = 0.80. Specifically, during visual feedback that understated
true rotation, pain-free range of motion was increased by
6% (95% confidence interval, or CI = [2%, 11%]); during
visual feedback that overstated true rotation, pain-free
range of motion decreased by 7% (95% CI = [3%, 11%]).
Therefore, our results show an overall effect of the
manipulation of 13%.
""")

# --------------------------------------------------------------------
# 1.  Function-calling schema  (machine-readable pipeline output)
# --------------------------------------------------------------------
tools = [
    {
        "type": "function",
        "function": {
            "name": "store_pipeline",
            "description": "Save the ordered list of analysis-pipeline steps",
            "parameters": {
                "type": "object",
                "properties": {
                    "steps": {
                        "type": "array",
                        "description": "Numbered analysis steps from data import to result compilation",
                        "items": {"type": "string"}
                    }
                },
                "required": ["steps"]
            }
        }
    }
]

# --------------------------------------------------------------------
# 2.  Prompt—using SYSTEM, DEVELOPER, ASSISTANT, USER roles
# --------------------------------------------------------------------
messages_client = [
    # ----- SYSTEM: overall persona & scope -----
    {
        "role": "system",
        "content": (
            f"{dev_job}  Focus exclusively on reproducing the statistical workflow "
            "described in the paper; ignore theoretical discussion or motivation."
        )
    },

    # ----- DEVELOPER: guard-rails & output specification -----
    {
        "role": "developer",
        "content": (
            "- Use only variables that appear in the provided dataset header.\n"
            "- Do not invent extra analyses, figures, or variable names.\n"
            "- Follow the numbered-step template exactly (see pipeline instructions).\n"
            "- When you are ready, CALL the `store_pipeline` function with the final list "
            "of steps; do not print the steps in plain text.\n"
            "- Make sure to identify column names by reading the header's of the dataset preview; "
            "those names must appear verbatim in your pipeline steps. Do make consideration of multi-indexing when applicable\n"
            "- Do not use placeholders like 'e.g.' or 'etc.'; every step must name a "
            "concrete R function. Also provide brief instructions with each step. "                
        )
    },

    # ----- ASSISTANT context blocks (large, one-time inputs) -----
    {"role": "assistant", "name": "article_excerpt",        "content": pdf_text},
    #
    #{"role": "assistant", "name": "figure_2_description",   "content": model_responses['figure_1']},
    #
    {"role": "assistant", "name": "dataset_preview",        "content": dataset_preview},
    #
    {"role": "assistant", "name": "dataset_summary",    "content": dataset_summary},
    #
    {"role": "assistant", "name": "dataset_codebook",    "content": codebook},
    #
    {"role": "assistant", "name": "additional_dataset_context", "content": add_dataset_context},

    # ----- USER: task + focal outcome specs -----
    {
        "role": "user",
        "content": (
            # which results to focus on 
            f"{hw_outcome_specs}\n\n"
            "Reference excerpt:\n"
            f"{article_ref}\n\n"
            # pipeline instructions
            "### Pipeline instructions\n"
            f"{pipeline_inst}\n\n"
            "### Action\n"
            ######### Add any further instructions here specific to the article #########
            "### Mandatory targets as per Reference excerpt (must appear in run_primary_analysis & compile_results steps)\n"
            "- Repeated measures ANOVA, show F-stat, df, p-value, partial eta-squared for all pairwise comparisons\n"
            "- t-test for understated and overstated visual feedback compared to accurate feedback. Provide the p-value and cohen's d or both. \n"
            f"- Provide the percent change in pain-free range of motion for understated and overstated visual feedback with 95% confidence interval. \n"

            # Generic instruction for prodcing pipeline
            "If you understand, produce the pipeline by calling the `store_pipeline` "
            "function with the ordered list of steps."
        )
    }
]

In [15]:
# --------------------------------------------------------------------
# Chat completion request  (o3-mini model)
# --------------------------------------------------------------------
response = client.chat.completions.create(
    model  = "o3-mini",
    tools  = tools,
    messages = messages_client
)

In [16]:
########## Stub for the dataset and the article #############
for m in messages_client:
    if m.get('name') in {"article_excerpt", "dataset_preview", "additional_dataset_context"}:
       m['content'] = f"(See {m['name']} provided earlier; doc_id = {m['name']})"

# Print the API response for review
import json

assistant_reply_1 = response.choices[0].message.tool_calls[0]
tool_call_args = json.loads(assistant_reply_1.function.arguments)
pipeline_steps = tool_call_args['steps']
print("Pipeline steps:\n")
#for i, step in enumerate(pipeline_steps, start = 1):
#    print(f"{i}. {step}\n")

pipeline_str = "\n".join(f"{step}"                                                  # use "{idx}. {step}" if steps don't print as numbered list
                         for idx, step in enumerate(pipeline_steps, start=1))

print(pipeline_str)

pipeline_msg = {"role": "assistant", "content": pipeline_str}

assistant_tool_msg_1 = {
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": assistant_reply_1.id,
            "type": "function",
            "function": {
                "name": assistant_reply_1.function.name,
                "arguments": json.dumps(tool_call_args)
            }
        }
    ]
}

tool_response_msg_1 = {
    "role": "tool",
    "tool_call_id": assistant_reply_1.id,
    "content": "OK",
}

messages_client.extend([assistant_tool_msg_1, tool_response_msg_1])

Pipeline steps:

1. import_libraries: Load required R packages using library(readr) for data import, library(dplyr) for data manipulation, library(tidyr) for reshaping data, library(ez) for repeated measures ANOVA, library(rstatix) for pairwise t-tests, and library(effectsize) for effect size calculations.
2. read_data: Read the dataset with read_csv('data.csv') ensuring that the header is correctly interpreted. This imports the variables Participant, DirectionofRotation, Condition1_Gain0.8, Condition2_Gain1, and Condition3_Gain1.2 among others.
3. clean_and_recode: Use dplyr functions to remove columns with all missing values (NormalisedDataUploadedToDataverse, RawData, NormalisedDataNotUploaded), and convert Participant to factor. No recoding for condition names is required as they already match the reported analysis.
4. isolate_target_subset: Select only the variables relevant for the primary outcome analysis by using dplyr::select to keep Participant, Condition1_Gain0.8, Condition2

In [ ]:
with open('Metrics_checker.txt', 'r') as file:
    metrics_check = file.read()

article_ref_wrapped = (
    "<<TARGET_RESULTS_START>>\n"
    f"{article_ref}\n"
    "<<TARGET_RESULTS_END>>"
)

# Codebook has been added here as it was available, can be removed if not available.
dataset_context = f"""
### Dataset preview and summary
{dataset_preview}

{dataset_summary}

{add_dataset_context}

"""


# ---------------------------------------------------------------------
# 1  System prompt  (persona + hard guard-rails)
# ---------------------------------------------------------------------
system_prompt = (
    "You are a senior data-science reviewer. Your sole task is to evaluate and "
    "improve an analysis pipeline for a scientific article. \n\n"
    "Allowed scope\n"
    "-------------\n"
    "Focus strictly on logical soundness, completeness, and coding feasibility. "
    "**only for the results inside the tags <<TARGET_RESULTS_START>> ... <<TARGET_RESULTS_END>>**\n"
    "Ignore language style and theoretical interpretation. \n\n"

    "Output format\n"
    "-------------\n"
    "Return your evaluation **as a JSON object** with two keys:\n"
    " - \"metric_summary\": a list of {\"metric\", \"score\", \"comment\"}\n"
    " - \"revised_pipeline\": a numbered list (array of strings) that fixes\n"
    "    any weaknesses you identified.\n"
    "Scores range 0-100.  Comment only when improvement is needed.  Do **not**\n"
    "output any text outside the JSON object."
)

# ---------------------------------------------------------------------
# 2  Build the user message (all context + instructions + pipeline)
# ---------------------------------------------------------------------
user_prompt = f"""
### Goal 
{hw_outcome_specs}

### Focus tags with the results to reproduce
{article_ref_wrapped}

### Mandatory targets as per Reference excerpt (must appear in run_primary_analysis & compile_results steps)\n
- Repeated measures ANOVA, show F-stat, df, p-value, partial eta-squared for all pairwise comparisons\n
- t-test for understated and overstated visual feedback compared to accurate feedback. Provide the p-value and cohen's d or both. \n
- Provide the percent change in pain-free range of motion for understated and overstated visual feedback with 95% confidence interval. \n


### Metrics for judging a pipeline
{metrics_check}

---

### Pipeline to review
{pipeline_str}

---

### Your tasks
1. Internally create your own pipeline (do **not** reveal it) to understand the
   target analysis.
2. Critique the reviewer’s pipeline using the supplied metrics (you may add
   well-defined metrics of your own, but explain them briefly in each comment).
3. Output **only** a JSON object with:
   - "metric_summary" – array of metric/score/comment triples  
   - "revised_pipeline" – the improved numbered pipeline

Remember: do not discuss numerical correctness of results, do not nit-pick
writing style, and do not output anything outside the JSON object.
"""




messages_checker = [
    # Context-only block (role=user or assistant both allowed)
    {"role": "user", "content": f"### Refer below for the excerpt from main article: \n{pdf_text}"},

    # Dataset preview (optional second context block)
    {"role": "user", "content": f"### Refer below for a summary of the dataset:\n{dataset_context}"},

    # Dataset codebook (optional second context block)
    {"role": "user", "content": f"### Refer below for a codebooke of the dataset: \n{codebook}"},

    # Table description (context-only block)
    #{"role": "user", "content": f"### Refer below for a description of the figure referred in the article: \n{model_responses['figure_1']}"},

    # Actual instruction block
    {"role": "user", "content": f" User prompt: {user_prompt}"}
]


In [18]:
response_h = checker.messages.create(
    model = 'claude-sonnet-4-20250514',
    system = f"{system_prompt}",
    messages = messages_checker,
    max_tokens = 4000
)



In [19]:
print(response_h.content[0].text)
checker_pipe = response_h.content[0].text

```json
{
  "metric_summary": [
    {
      "metric": "Specification Completeness (SC)",
      "score": 75,
      "comment": "Most steps are well-specified but Step 3 mentions removing columns without specifying exact column names, and Step 4 lacks clarity on handling DirectionofRotation variable which appears in the dataset but not in the selection."
    },
    {
      "metric": "Modularity Index (MI)",
      "score": 85,
      "comment": "Good separation of concerns with atomic steps, though Step 3 combines cleaning and recoding operations that could be separated for better modularity."
    },
    {
      "metric": "Parameter Explicitness Ratio (PER)",
      "score": 60,
      "comment": "Missing explicit specification of alpha level (0.05), degrees of freedom calculation method, and specific function parameters for ezANOVA configuration."
    },
    {
      "metric": "Provenance Specification Level (PSL)",
      "score": 70,
      "comment": "Good coverage of What and How elements, 

In [21]:
# Extract and convert the revised pipeline into string format

import re

raw = response_h.content[0].text

json_txt = re.sub(r"^```json|```$", "", raw.strip(), flags=re.MULTILINE).strip()

review_obj = json.loads(json_txt)

revised_pipe = review_obj["revised_pipeline"]

revised_pipe_str = "\n".join(
    f"{step}" for idx, step in enumerate(revised_pipe)                     #{idx+1}. {step} if steps don't print as numbered list
)

print(revised_pipe_str)

1. import_libraries: Load required R packages: library(readr), library(dplyr), library(tidyr), library(ez), library(rstatix), library(effectsize), and library(broom) for statistical output formatting.
2. read_data: Import dataset using read_csv('data.csv') with explicit column specification to handle the 48 observations (24 participants × 2 directions) structure.
3. clean_data: Remove columns with all missing values (NormalisedDataUploadedToDataverse, RawData, NormalisedDataNotUploaded) using select(-all_of(c('NormalisedDataUploadedToDataverse', 'RawData', 'NormalisedDataNotUploaded'))).
4. recode_variables: Convert Participant to factor and ensure proper data types for analysis variables using mutate(Participant = as.factor(Participant)).
5. aggregate_directions: Calculate mean pain-free range of motion across left and right rotation directions for each participant and condition using group_by(Participant) and summarise across Condition1_Gain0.8, Condition2_Gain1, and Condition3_Gain1

Or you can add a prompt here for either letting GPT make the suggested changes by you and anthropic or the user can make changes on their own. 


In [ ]:
# ---------------------------------------------------------------
# 0   Helper: function-calling schema
# ---------------------------------------------------------------
tools_setup = [
    {
        "type": "function",
        "function": {
            "name": "submit_setup_plan",
            "description": "Return library list, dataset understanding, and cleaning steps",
            "parameters": {
                "type": "object",
                "properties": {
                    "libraries": {
                        "type": "array",
                        "description": "Each entry: package name and a one-line purpose",
                        "items": {
                            "type": "object",
                            "properties": {
                                "package": {"type": "string"},
                                "purpose": {"type": "string"}
                            },
                            "required": ["package", "purpose"]
                        }
                    },
                    "dataset_overview": {
                        "type": "array",
                        "description": "Schema summary (col name, inferred type, comment)",
                        "items": {
                            "type": "object",
                            "properties": {
                                "column":  {"type": "string"},
                                "type":    {"type": "string"},
                                "comment": {"type": "string"}
                            },
                            "required": ["column", "type", "comment"]
                        }
                    },
                    "final_pipeline": {
                        "type": "array",
                        "description": "Numbered list of pipeline steps (strings)",
                        "items": {"type": "string"}
                    },
                    "patch_applied": {
                        "type": "object",
                        "description": "Echo back exactly what USER_EDITS changed",
                        "properties": {
                            "steps_removed": {"type": "array", "items": {"type": "string"}},
                            "steps_added":   {"type": "array", "items": {"type": "string"}},
                            "steps_modified": {"type": "array", "items": {"type": "string"}}
                        },
                        "required": []
                    },
                    "cleaning_pipeline": {
                        "type": "array",
                        "description": "Numbered steps needed before analysis, or empty if none",
                        "items": {"type": "string"}
                    },
                    "needs_cleaning": {
                        "type": "boolean",
                        "description": "True if any cleaning is required"
                    }
                },
                "required": ["libraries", "dataset_overview", "cleaning_pipeline", "needs_cleaning"]
            }
        }
    }
]

# ---------------------------------------------------------------
# 1   Developer guard-rail 
# ---------------------------------------------------------------
third_dev = (
    "You are now preparing to write R code for the FINAL analysis pipeline.\n"
    "- Ignore any other pipeline apart from the one provided in this message. \n"
    "- Task today: **plan**, do not write code.\n"
    "- Ignore any pipeline except the one shown above USER_EDITS.\n"
    "- Apply ONLY the edits listed in USER_EDITS. If USER_EDITS is empty/absent, "
    "make **no changes**.\n"
    "- **Return `final_pipeline` as the COMPLETE numbered list after edits; "
    "if no edits, return the original list unchanged.**\n"
    "- Record what changed in `patch_applied` (steps_added / steps_removed / steps_modified).\n"
    "- Output via the `submit_setup_plan` function only; no prose outside JSON.\n"
    "- List only R packages actually needed for the pipeline (e.g., readr, dplyr, stats).\n"
    "- For each package give one short clause on what it will do.\n"
    "- Infer dataset schema from the preview; reference columns exactly as named.\n"
    "- Decide whether cleaning/wrangling is required; if yes, give a numbered pipeline "
    "(e.g., convert factors, handle NAs). If no, explain via `needs_cleaning=false` and "
    "leave `cleaning_pipeline` empty."
)

# ---------------------------------------------------------------
# 2   User instruction 
# ---------------------------------------------------------------
third_user = (
    f"Here is the **first draft of analysis pipeline** that is approved:\n{revised_pipe_str}\n\n"
##### Add here any changes or addition to the pipeline that the model should consider



   "### USER_EDITS \n"
#   "Consider following updates to the pipeline:\n"
#   "1. \n"
#   "Consider removing the following steps from the pipeline:\n"
#   "1. \n"
#################### OR ####################
   "No updates to the revised pipeline are needed at this time.\n"
   "### END_EDITS\n\n"

#    "Dataset preview:\n"
#    f"{dataset}\n\n"
    "Please analyse the pipeline and dataset provided earlier as instructed."
)

# ---------------------------------------------------------------
# 3   Append to running history & call
# ---------------------------------------------------------------
messages_client.extend([
    {"role": "developer", "content": third_dev},
    {"role": "user",      "content": third_user}
])



In [23]:
response_2 = client.chat.completions.create(
    model     = "o3-mini",
    messages  = messages_client,
    tools     = tools_setup,
)

In [ ]:
assistant_reply_2 = response_2.choices[0].message.tool_calls[0]
tool_call_args_2 = json.loads(assistant_reply_2.function.arguments)

# ------------------------------------------------------------------
#  Print for your own review 
# ------------------------------------------------------------------

libs_block = "\n".join(
    f"- {item['package']}: {item['purpose']}"
    for item in tool_call_args_2["libraries"]
)

schema_block = "\n".join(
    f"- {col['column']} ({col['type']}): {col['comment']}"
    for col in tool_call_args_2["dataset_overview"]
)

pipeline_fin = tool_call_args_2["final_pipeline"]
final_pipeline_block = "\n".join(pipeline_fin)                  

patch = tool_call_args_2.get("patch_applied", {})               # may be empty if no edits were made
added_block = "\n".join(f" -{s}" for s in patch.get("steps_added", [])) or " none (no edits)"
removed_block = "\n".join(f" -{s}" for s in patch.get("steps_removed", [])) or " none (no edits)"
modified_block = "\n".join(f" -{s}" for s in patch.get("steps_modified", [])) or " none (no edits)"

if tool_call_args_2["needs_cleaning"]:
    clean_steps = [s for s in tool_call_args_2["cleaning_pipeline"] if s.strip()]
    cleaning_block = "\n".join(
        f"{step}"
        for i, step in enumerate(clean_steps, start=1)
    )
    cleaning_block = "Cleaning pipeline:\n" + cleaning_block
else:
    cleaning_block = "Cleaning pipeline: none (data ready)"

summary_str = (
    "### Libraries\n"
    f"{libs_block}\n\n"
    "### Dataset overview\n"
    f"{schema_block}\n\n"
    f"{cleaning_block}\n\n"
    f"Cleaning required: {'yes' if tool_call_args_2['needs_cleaning'] else 'no'}\n\n"
    "### Final pipeline\n"
    f"{final_pipeline_block}\n\n"
    "### Edits applied\n"
    f"Steps added:\n{added_block}\n\n"
    f"Steps removed:\n{removed_block}\n\n"
    f"Steps modified:\n{modified_block}\n\n"
)

print(summary_str)

# ------------------------------------------------------------------
#  Append assistant-tool message to the running history
# ------------------------------------------------------------------

assistant_tool_msg_2 = {
    "role": "assistant",
    "content": None,                
    "tool_calls": [
        {
            "id":        assistant_reply_2.id,
            "type":      "function",
            "function": {
                "name":       assistant_reply_2.function.name,
                "arguments":  assistant_reply_2.function.arguments
            }
        }
    ]
}

    

messages_client.extend([
    assistant_tool_msg_2,
    { "role": "tool",
      "tool_call_id": assistant_reply_2.id,
      "content": "OK"
    },
    { "role": "assistant",               
      "name": "pipeline_v2",
      "content": "FINAL PIPELINE AFTER EDITS:\n" + final_pipeline_block
    }
])


### Libraries
- readr: Import CSV data using read_csv
- dplyr: Perform data manipulation and cleaning
- tidyr: Reshape data (pivot_longer) for repeated measures analysis
- ez: Conduct repeated measures ANOVA and test sphericity
- rstatix: Perform pairwise paired t-tests with Bonferroni correction
- effectsize: Calculate effect sizes such as Cohen's d and partial eta-squared
- broom: Format statistical output for reporting

### Dataset overview
- Participant (float64): Participant numbers (1 to 24); to be converted to factor
- DirectionofRotation (float64): Indicates left (1) or right (2) rotation
- Condition1_Gain0.8 (float64): Pain-free range of motion for gain 0.8 condition
- Condition2_Gain1 (float64): Pain-free range of motion for gain 1 condition (accurate feedback)
- Condition3_Gain1.2 (float64): Pain-free range of motion for gain 1.2 condition
- NormalisedDataUploadedToDataverse (float64): Column with all missing values to be removed
- RawData (float64): Column with all missing 

In [ ]:
# ------------------------------------------------------------------
# 0   Function schema: model asks clarifications or confirms ready
# ------------------------------------------------------------------
tools_clarify = [
    {
        "type": "function",
        "function": {
            "name": "ask_or_confirm",
            "description": (
                "If you need clarifications about dataset assumptions or logical gaps, "
                "list them. Otherwise confirm you are ready to code."
                "Also echo back any library/dataset edits requested by the user."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "needs_clarification": {
                        "type": "boolean",
                        "description": "True => you still have questions"
                    },
                    "questions": {
                        "type": "array",
                        "description": "List of clarification questions (ignored if flag is false)",
                        "items": {"type": "string"}
                    },
                    "ready_message": {
                        "type": "string",
                        "description": "Short confirmation like 'All clear, ready to code'; empty if questions exist"
                    },
                    "patch_mentioned":{
                        "type": "boolean",
                        "description": "True => user edits were mentioned in the USER_EDITS block; empty if no edits were requested"
                    },
                    "patch": {                                    
                        "type": "object",
                        "description": "Edits explicitly requested by user",
                        "properties": {
                            "libraries_add":  {
                                "type": "array",
                                "items": {"type": "string"},
                                "description": "R packages to ADD"
                            },
                            "libraries_drop": {
                                "type": "array",
                                "items": {"type": "string"},
                                "description": "R packages to REMOVE"
                            },
                            "dataset_notes":  {
                                "type": "string",
                                "description": "Update to dataset_overview (one short paragraph)"
                            }
                        },
                        "required": []
                    }
                },
                "required": ["needs_clarification", "questions", "ready_message", "patch"]
            }
        }
    }
]

# ------------------------------------------------------------------
# 1   Developer guard-rail
# ------------------------------------------------------------------
clarify_dev = (
    "We have finalised the analysis pipeline and selected libraries.\n"
    "User may ask for edits to libraries or dataset notes.\n"
    "- Apply ONLY the edits listed in the USER_EDITS block below (no new changes).\n"
    "- Do NOT write any R code yet.\n"
    "- Think about dataset assumptions and logical gaps.\n"
    "- If you need additional info, call `ask_or_confirm` with "
    "needs_clarification=true and a list of concise questions.\n"
    "- If everything is clear, call `ask_or_confirm` with "
    "needs_clarification=false and a short ready_message.\n"
    "- Output nothing outside the tool call."
)

# ------------------------------------------------------------------
# 2   User prompt
# ------------------------------------------------------------------
clarify_user = (
    "Refer to the pipeline in the message named 'pipeline_v2' for all steps.\n"
    "I agree with the final pipeline and the library list.\n"
    "Before you start coding, ask me any clarifications you need about:\n"
    " - Your assumptions on dataset cleaning/wrangling\n"
    " - Any logical or reasoning gaps you perceive\n"
    "If no clarifications are needed, just confirm you are ready."
##### Additional prompts to consider for any changes that need to be suggested in model understanding. #####
   "### USER_EDITS \n"
#   "Add libraries: \n"
#   "Delete libraries: \n"
#   "DATASET NOTES: \n"
########### OR if no change add none ############
    "No edits to the libraries or dataset notes are needed at this time.\n"
   "### END_EDITS \n"
)

# ------------------------------------------------------------------
# 3   Extend history & call 
# ------------------------------------------------------------------
messages_client.extend([
    {"role": "developer", "content": clarify_dev},
    {"role": "user",      "content": clarify_user}
])


In [26]:
response_3 = client.chat.completions.create(
    model    = "o3-mini",
    messages = messages_client,
    tools    = tools_clarify
)


In [27]:
assistant_reply_3 = response_3.choices[0].message.tool_calls[0]
tool_call_args_3 = json.loads(assistant_reply_3.function.arguments)

# ------------------------------------------------------------------
#  Print for your own review 
# ------------------------------------------------------------------

patch_flag = tool_call_args_3.get("patch_mentioned", False)

if patch_flag:
    p = tool_call_args_3["patch"]
    add_block  = "\n".join(f"- {lib}" for lib in p["libraries_add"])  or " none "
    drop_block = "\n".join(f"- {lib}" for lib in p["libraries_drop"]) or " none "
    dataset_notes_block = p["dataset_notes"].strip() or " none "
    patch_str = (
        "### User edits\n"
        f"**Libraries to add**:\n{add_block}\n\n"
        f"**Libraries to drop**:\n{drop_block}\n\n"
        f"**Dataset notes**:\n{dataset_notes_block}\n\n"
    )
else:
    patch_str = "### User edits\nNone – no changes requested\n\n"

if tool_call_args_3["needs_clarification"]:
    questions_block = "\n".join(f"- {q}" for q in tool_call_args_3["questions"])
    clarify_str = (
        "### Clarifications needed\n"
        f"{questions_block}\n"
    )
else:
    clarify_str = (
        "### Clarifications needed\n"
        "None – all clear \n\n"
        f"Ready message: {tool_call_args_3['ready_message']}"
    )

print(f"{patch_str}")
print(clarify_str)

# ------------------------------------------------------------------
#  Append assistant-tool message to the running history
# ------------------------------------------------------------------

assistant_tool_msg_3 = {
    "role": "assistant",
    "content": None,                
    "tool_calls": [
        {
            "id":        assistant_reply_3.id,
            "type":      "function",
            "function": {
                "name":       assistant_reply_3.function.name,
                "arguments":  assistant_reply_3.function.arguments
            }
        }
    ]
}


tool_response_msg_3 = {
    "role": "tool",
    "tool_call_id": assistant_reply_3.id,
    "content": "OK",
}

messages_client.extend([assistant_tool_msg_3, tool_response_msg_3])

### User edits
None – no changes requested


### Clarifications needed
None – all clear 

Ready message: All clear, ready to code.


In [ ]:
# ------------------------------------------------------------------
# 1.  Developer guard-rail – force code-only output
# ------------------------------------------------------------------
code_dev = (
    "You have all clarifications.  Produce the FINAL R script.\n"
    "Output rules:\n"
    " - Return exactly ONE fenced code block: ```r ... ```\n"
    " - Begin with necessary `library()` calls.\n"
    " - Inline comments (#) must map to the numbered pipeline steps.\n"
    " - Use only variables present in the dataset header.\n"
    " - Saving output:\n"
    "     – If the user has asked to save a file, include the write step\n"
    "       (e.g., `write.csv()` or `saveRDS()`).\n"
    "     – Otherwise, print to console.\n"
    " - Plotting:\n"
    "     - You may generate a plot **only** if the article reference reports a figure\n"
    "       as part of the requested results.\n"
    "     – Use ggplot2; save with `ggsave()` if the user asked to save files.\n"
    " - Do NOT write prose outside the code block.\n"
)

# ------------------------------------------------------------------
# 2.  User message – your clarifications
# ------------------------------------------------------------------
code_user = (
    #"Clarification answers:\n"
    #"1. Yes the exclusion criteria metnioned by you is correct. \n"
    #"2. No other cleaning steps are required.\n"

    "No clarification was required by you.\n\n"

    #"Additional instructions:\n"
    ############# Instruction to save results csv ###############
    #" - Please **save** the final summary table as 'summary_table.csv'.\n"
    ############# Instrctions to create and save plots ################
    #" - Also recreate the plot similar to the one shared earlier. Create it using the dataset provided,\n"
    #"  and save it as 'plot.png'.\n"
    ####################################################################
    "Now produce the code."
)

# ------------------------------------------------------------------
# 3.  Extend history & call 
# ------------------------------------------------------------------
messages_client.extend([
    {"role": "developer", "content": code_dev},
    {"role": "user",      "content": code_user}
])



In [29]:
response_4 = client.chat.completions.create(
    model="o3-mini",
    messages=messages_client,
)
print("Response for the fourth prompt:\n")
print(response_4.choices[0].message.content)
assistant_reply_4 = response_4.choices[0].message.content

Response for the fourth prompt:

```r
# 1. import_libraries: Load required R packages.
library(readr)      # Import CSV data
library(dplyr)      # Data manipulation and cleaning
library(tidyr)      # Data reshaping
library(ez)         # Repeated measures ANOVA and assumption tests
library(rstatix)    # Pairwise paired t-tests and effect size calculations
library(effectsize) # Calculation of effect sizes (Cohen's d, partial eta²)
library(broom)      # Formatting statistical output

# 2. read_data: Import the dataset with explicit column handling.
data <- read_csv("data.csv", col_types = cols(
  Participant = col_double(),
  DirectionofRotation = col_double(),
  NormalisedDataUploadedToDataverse = col_double(),
  Condition1_Gain0.8 = col_double(),
  Condition2_Gain1 = col_double(),
  Condition3_Gain1.2 = col_double(),
  RawData = col_double(),
  Point8 = col_double(),
  One = col_double(),
  Onepoint2 = col_double(),
  NormalisedDataNotUploaded = col_double(),
  Point.8 = col_double(),
 

In [ ]:
# ------------------------------------------------------------------
# 1  System prompt  – guard-rails 
# ------------------------------------------------------------------
system_prompt_1 = f"""
You are a senior data-science reviewer. Checking R code based on below metrics, considering the pipeline provided by the coder.

Allowed scope
-------------
- Logical correctness (does each step implement the stated pipeline?)
- function compatibility with the libraries used (e.g. does the function exist, is it used correctly)  
- Data-safety issues (e.g. NA handling if NAs possible, factor coercion, wrong column names)  
- Runtime efficiency (vectorised ops, avoiding unnecessary copies)  
- Ignore spelling, commenting style, or aesthetic refactors  
- Do not add new tests, plots, or validations unless the **pipeline explicitly requires them**.
- The user may provide error messages or warnings encountered during execution, with the specific code chunks. 
  Take this into account when reviewing and revising the code. Provide feedback on that as well.
- If no error or warning messages are provided ignore that block in output.

Fault vs. nit-pick
------------------
A *fault* is any construct that could give wrong numbers, crash, or be slow for the stated dataset.
Examples to flag:
- Forgetting `na.rm = TRUE` when summarising numeric data that may contain NAs  
- Using a function that does not exist in the loaded libraries
- Referencing a column not in the dataset  
- Double-reading the same file or using a redundant loop

Output format (MUST follow exactly)
-----------------------------------
### Feedback
- Metric: <name> — <concise remark>  (only if a real fault or big efficiency gain)

### Revised_Code
```r
# complete runnable script in R
```
### Change_Log
- <what you changed & why> (reference a Feedback bullet)

### Error correction handling
- <if the user provided error messages, explain how you addressed them in the revised code. Otherwise, mention that no errors were encountered by user.>
"""

# ------------------------------------------------------------------
# 2  User prompt – supply code to review
# ------------------------------------------------------------------

user_prompt_1 = (f"""
Here is the peer-reviewer's R code:
{assistant_reply_4}
Remember: comment only on logic/runtime faults or big efficiency wins.
Do not invent extra analyses, plots, or style tweaks.\n
"""

################ Use this part when running again to make fixes ###################################### 
################ and comment out the first text box above with {assistant reply}######################
#f"""
#Below is the final R code:
#{checker_code}
#Remember: Your objective is to correct logical faults and fix errors in code chunks provided by user or updates suggested by user.
#Do not invent extra analyses, plots, or style tweaks.\n
#"""
######################################################################################################


f"""
### Error / warning encountered:
# Code chunk:

# Error chunk:
No error or warning messages were encountered by the user.
"""
##################### Add for error handling by running this a second time ######################
"""
##While reviewing the code, consider the following updates to the pipeline:

##### USER_UPDATES
Instead of averaging the range score for each side of rotation, consider them separately. 
Also, for calculating the shapiro test you would need to create a seperate subset without the 'Conditions2_Gain1' as tis only has 1 value.
##### END_USER_UPDATES
#"""
###### When mentioning updates do ensure to mention any suspected results that may go missing.
)

# ------------------------------------------------------------------
# 3.  Add final pipeline & call 
# ------------------------------------------------------------------

messages_checker = ([{"role": "assistant", "content": f"{final_pipeline_block}"},
    {"role": "user", "content": f"{user_prompt_1}"}])

In [31]:
messages_checker

[{'role': 'assistant',
  'content': "1. import_libraries: Load required R packages: library(readr), library(dplyr), library(tidyr), library(ez), library(rstatix), library(effectsize), and library(broom) for statistical output formatting.\n2. read_data: Import dataset using read_csv('data.csv') with explicit column specification to handle the 48 observations (24 participants × 2 directions) structure.\n3. clean_data: Remove columns with all missing values (NormalisedDataUploadedToDataverse, RawData, NormalisedDataNotUploaded) using select(-all_of(c('NormalisedDataUploadedToDataverse', 'RawData', 'NormalisedDataNotUploaded')).\n4. recode_variables: Convert Participant to factor and ensure proper data types for analysis variables using mutate(Participant = as.factor(Participant)).\n5. aggregate_directions: Calculate mean pain-free range of motion across left and right rotation directions for each participant and condition using group_by(Participant) and summarise across Condition1_Gain0.8

In [33]:
response_h_1 = checker.messages.create(
    model = 'claude-sonnet-4-20250514',
    system = f"{system_prompt_1}",
    messages = messages_checker,
    max_tokens = 5000
)

In [34]:
print(response_h_1.content[0].text)
checker_code = response_h_1.content[0].text

### Feedback
- Logical correctness — Pipeline step 5 averages across rotation directions, but user update requests treating them separately
- Data-safety issues — Shapiro-Wilk test on Condition2_Gain1 may fail if it has only 1 unique value per the user update
- Logical correctness — User update indicates Condition2_Gain1 has only 1 value, which would make it inappropriate as a baseline for comparisons

### Revised_Code
```r
# 1. import_libraries: Load required R packages.
library(readr)      # Import CSV data
library(dplyr)      # Data manipulation and cleaning
library(tidyr)      # Data reshaping
library(ez)         # Repeated measures ANOVA and assumption tests
library(rstatix)    # Pairwise paired t-tests and effect size calculations
library(effectsize) # Calculation of effect sizes (Cohen's d, partial eta²)
library(broom)      # Formatting statistical output

# 2. read_data: Import the dataset with explicit column handling.
data <- read_csv("data.csv", col_types = cols(
  Participa

Error correction can be run through anthropic again by reprompting - as mentioned in the prompt section above